# First RAG, 검색 결과를 LLM 답변으로 연결하기
- 문서를 가져와서 청크로 나누고, 질문과 가까운 청크를 검색한 뒤 LLM 프롬프트에 넣어 답변을 생성함

- 문서 → split → embed/search → prompt → LLM 답변


## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [ ]:
# 필요한 라이브러리 설치
# uv add -qU langchain langchain-openai langchain-text-splitters python-dotenv numpy


## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


## 2. 문서 가져오기 + 검색기 만들기


In [2]:
SAMPLE = """## 우주지사 근무 규정

### 1. 화성 지사 출근
화성 지사 근무자는 지구 표준시 기준 오전 9시까지 메타버스 출근 체크를 완료해야 한다. 출근 체크는 사내 근태 시스템의 "화성 지사 원격 출근" 메뉴에서 진행하며, 위치 인증과 생체 인증을 모두 통과해야 정상 출근으로 인정된다.

산소 농도 경고가 발생한 경우에는 출근 체크보다 안전 확인 보고서를 먼저 제출해야 한다. 안전 확인 보고서에는 현재 위치, 산소 농도 수치, 근무 가능 여부, 대피 필요 여부를 기록해야 한다. 산소 농도가 기준치 이하로 10분 이상 유지되면 해당 근무자는 자동으로 비상 대기 상태로 전환된다.

화성 모래폭풍 경보가 발령된 날에는 지사장이 재택근무 전환 여부를 공지한다. 재택근무로 전환된 경우에도 오전 10시까지 업무 계획을 팀 채널에 공유해야 하며, 긴급 연락을 받을 수 있도록 메신저 상태를 온라인으로 유지해야 한다.

### 2. 우주복 장비 대여
외부 기지 이동 시 우주복, 산소팩, 자기부착 신발을 반드시 대여해야 한다. 장비 대여는 이동 예정 시간 최소 1시간 전까지 장비 관리 시스템에서 신청해야 하며, 신청서에는 이동 목적, 예상 이동 경로, 복귀 예정 시간을 입력해야 한다.

우주복은 사용 후 18시까지 장비실에 반납해야 한다. 반납 시에는 외부 손상 여부, 산소 밸브 상태, 통신 모듈 작동 여부를 점검표에 기록해야 한다. 우주복에 균열이 있거나 통신 모듈 오류가 발견되면 즉시 장비 담당자에게 보고해야 하며, 임의로 수리해서는 안 된다.

산소팩 잔량이 20% 미만이면 즉시 교체 신청을 해야 한다. 산소팩 잔량이 10% 이하로 떨어진 상태에서 외부 이동을 계속하는 것은 중대한 안전 규정 위반으로 간주된다. 자기부착 신발은 기지 외부에서는 항상 활성화해야 하며, 실내 복귀 후에는 바닥 손상을 방지하기 위해 비활성화해야 한다.

### 3. 화성 회의실 예약
화성 회의실은 최소 2시간 전까지 예약해야 한다. 예약은 사내 캘린더의 "화성 지사 회의실" 메뉴에서 진행하며, 회의 목적, 참석자 수, 예상 소요 시간, 필요한 장비를 함께 입력해야 한다. 회의실은 기본 1시간 단위로 예약할 수 있으며, 3시간을 초과하는 회의는 지사장 승인이 필요하다.

6명 이상 참석하는 회의는 산소 소비량 계산을 위해 참석자 명단을 함께 등록해야 한다. 참석자가 외부 방문자인 경우에는 방문 목적과 소속 기관을 추가로 입력해야 하며, 보안 구역 회의실은 외부 방문자 예약이 제한된다.

회의 시작 10분 전까지 입실하지 않으면 예약은 자동 취소될 수 있다. 회의 종료 후에는 공용 화면, 홀로그램 프로젝터, 산소 조절 장치를 초기 상태로 되돌려야 한다. 회의 중 산소 농도 알림이 발생하면 회의를 즉시 중단하고, 참석자는 가장 가까운 안전 구역으로 이동해야 한다.
"""

print(f"문서 길이: {len(SAMPLE)} 자")

문서 길이: 1378 자


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import numpy as np

# 1. 텍스트 분할기 생성
# 긴 문서를 일정한 크기의 청크(chunk)로 나누기 위한 splitter를 만든다.
# chunk_size=200은 청크 하나의 최대 길이를 의미하고,
# chunk_overlap=20은 청크끼리 20글자 정도 겹치게 해서 문맥 손실을 줄인다.
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 20
)

# 2. SAMPLE 문자열을 여러 개의 청크로 분할
# SAMPLE이라는 긴 문자열을 여러 개의 작은 텍스트 조각으로 나눈다.
# 결과인 chunks는 문자열 리스트이다.
chunks = splitter.split_text(SAMPLE)

# 3. 임베딩 모델 생성
# 각 청크의 의미를 담은 숫자 벡터로 변환할 때 사용할 임베딩 모델을 만든다.
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small'
)

# 4. 청크들을 임베딩 벡터로 변환
# chunks에 들어 있는 텍스트들을 각각 숫자 벡터로 변환한다.
# np.array로 감싸서 벡터 계산을 하기 쉬운 2차원 배열 형태로 만든다.
chunk_vectors = np.array(embeddings.embed_documents(chunks))

# 5. 청크 벡터 정규화
# 각 벡터의 길이를 1로 맞춘다.
# 이렇게 해두면 dot product 계산 결과가 코사인 유사도처럼 동작한다.
chunk_vectors_n = chunk_vectors / np.linalg.norm(chunk_vectors,
                                                 axis=-1,
                                                 keepdims=True)


# 6. 검색 함수 정의
# 사용자의 질문을 임베딩한 뒤,
# 기존 청크 벡터들과 유사도를 계산해서 가장 비슷한 청크 k개를 반환한다.
def retrieve(query: str, k: int = 3):
    # 6-1. 질문을 임베딩 벡터로 변환
    q_vec = embeddings.embed_query(query)
    # 6-2. 질문 벡터도 정규화
    q_vec_n = np.array(q_vec) / np.linalg.norm(q_vec)
    # 6-3. 질문 벡터와 모든 청크 벡터의 유사도 계산
    sims = chunk_vectors_n @ q_vec_n
    # 6-4. 유사도가 높은 순서대로 상위 k개 인덱스 선택
    top_idx = np.argsort(sims)[::-1][:k]
    # 6-5. 선택된 인덱스에 해당하는 원본 청크 반환
    return [chunks[i] for i in top_idx]

## 3. LCEL 로 단순 RAG 체인 구성
 - LangChain Expression Language 의 `|` 파이프로 검색 결과 → 프롬프트 → LLM → 문자열 파서 연결


In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0)

RAG_PROMPT = ChatPromptTemplate.from_messages([
    (
        'system'
        '너는 우주지사 근무 규정 QA assistant이다. 아래 참고 자료만 근거로 한국어로 답하라.'
        '''참고 자료에 없으면 "자료에서 확인할 수 없습니다."라고 답하라.'''

    ),
    ('user', '참고 자료:\n{context}\n\n 질문:{question}')
])


def format_docs(docs: list[str]) -> str:
    return "\n\n".join(f"[{i+1}] {doc}" for i, doc in enumerate(docs))

# 사용자 질문 -> retrieve()로 관련 청크 검색 -> RAG_PROMPT에 context, question 삽입 -> llm이 답변 생성 -> outputparser가 반환
rag_chain = (
    {
        'context' : lambda x: format_docs(retrieve(x['question'], k=3)),
        'question' : lambda x: x['question']
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

In [8]:
question = '우주복은 언제 반납?'

print(f'질문:{question}')
answer = rag_chain.invoke({'question': question})
print(f'답변:{answer}')

질문:우주복은 언제 반납?
답변:우주복은 사용 후 18시까지 장비실에 반납해야 합니다.


## 4. 실행


In [10]:
# question = '우주복은 언제 반납?'
# question = '화성 지사 출근 체크는 몇시까지 해야 되나?'

question = '우주에서 식사 메뉴는 뭐야?'

print(f'질문:{question}')
answer = rag_chain.invoke({'question': question})
print(f'답변:{answer}')

질문:우주에서 식사 메뉴는 뭐야?
답변:자료에서 확인할 수 없습니다.


## 5. 답의 출처 같이 반환하기

- 답변만 보여주면 사용자가 근거를 확인하기 어려움
- 검색된 청크를 함께 반환하면 RAG 답변의 신뢰도를 점검할 수 있음

In [ ]:
def answer_from_docs(question: str, docs: list[str]) -> str:
    chain = RAG_PROMPT | llm | StrOutputParser()
    return chain.invoke({"context": format_docs(docs), "question": question})


def rag_with_sources(question: str, k: int = 3):        # 1. 질문을 받아서
    docs = retrieve(question, k=k)                      # 2. 관련 문서 검색
    answer = answer_from_docs(question, docs)           # 3. 검색된 문서를 근거로 답변 생성
    return {                                            # 4. 질문, 답변, 출처 청크를 함께 반환
        "question": question,
        "answer": answer,
        "sources": docs,
    }


result = rag_with_sources("산소팩 잔량이 20% 미만이라면?")

print(f"질문: {result['question']}")
print(f"답변: {result['answer']}")
print("\n=== 출처 청크 ===")
for i, source in enumerate(result["sources"], start=1):
    print(f"[{i}] {source[:120]}")


질문:  
답변: 질문을 입력해 주시면 참고 자료를 바탕으로 답변해 드리겠습니다.

=== 출처 청크 ===
[1] 화성 모래폭풍 경보가 발령된 날에는 지사장이 재택근무 전환 여부를 공지한다. 재택근무로 전환된 경우에도 오전 10시까지 업무 계획을 팀 채널에 공유해야 하며, 긴급 연락을 받을 수 있도록 메신저 상태를 온라인으로 유
[2] 산소팩 잔량이 20% 미만이면 즉시 교체 신청을 해야 한다. 산소팩 잔량이 10% 이하로 떨어진 상태에서 외부 이동을 계속하는 것은 중대한 안전 규정 위반으로 간주된다. 자기부착 신발은 기지 외부에서는 항상 활성화해
[3] 산소 농도 경고가 발생한 경우에는 출근 체크보다 안전 확인 보고서를 먼저 제출해야 한다. 안전 확인 보고서에는 현재 위치, 산소 농도 수치, 근무 가능 여부, 대피 필요 여부를 기록해야 한다. 산소 농도가 기준치 이


## 6. "자료에 없으면 모르겠습니다" 가 왜 중요한가
- RAG 의 가장 큰 위험은 **환각** 인데, 환각은 검색 결과와 무관한 답을 LLM 이 지어내는 현상
- 시스템 프롬프트의 `"자료에 없으면 확인할 수 없다고 답하라"` 같은 한 줄이 환각을 크게 줄임


### 환각 비교, 같은 "날씨" 질문

In [12]:
# 위 RAG_PROMPT는 '자료에 없으면 확인할 수 없다'를 명시
rag_strict = rag_chain.invoke({'question':'오늘 날씨 어때?'})

# 자료가 없어도 일반 지식으로 답하게 함
LOOSE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "참고 자료가 있으면 활용하고, 없으면 일반 지식으로 답해"),
    ("user", "참고 자료:\n{context} \n\n 질문:{question}"),
])

loose_chain = (
    {
        "context": lambda x: format_docs(retrieve(x["question"], k=3)),
        "question": lambda x: x["question"],
    }
    | LOOSE_PROMPT
    | llm
    | StrOutputParser()
)

rag_loose = loose_chain.invoke({"question": "오늘 날씨 어때?"})

print(f"엄격 RAG: {rag_strict}")
print()
print(f"느슨 RAG: {rag_loose}")


엄격 RAG: 자료에서 확인할 수 없습니다.

느슨 RAG: 참고 자료에 오늘 화성의 날씨에 대한 정보는 포함되어 있지 않습니다. 일반적으로 화성에서는 모래폭풍이 발생할 수 있으며, 모래폭풍 경보가 발령되면 재택근무 전환 여부가 공지됩니다. 오늘 화성의 구체적인 날씨 상황을 확인하려면 화성 지사 내 공식 기상 관측 시스템이나 관련 공지를 참고하시기 바랍니다.


## 7. 정리

- 문서를 청크로 나눈 뒤 임베딩해 검색기를 만들었습니다.
- LCEL 로 `검색 → 프롬프트 → LLM → 출력 파서` 체인을 구성했습니다.
- 답변과 출처 청크를 함께 반환했습니다.
- 다음 실습에서는 numpy 검색기 대신 Chroma vectorDB 를 사용합니다.

## [실습]

1. `retrieve()`의 `k` 값을 1, 3, 5 로 바꿔 답변 차이를 비교합니다.
2. `SAMPLE`에 새 규정을 추가하고 검색 결과가 바뀌는지 확인합니다.
3. 답에 "[1] 출처: ..." 형태로 인용 마커를 자동 삽입.
4. 프롬프트의 `자료에서 확인할 수 없습니다` 문구를 제거하면 자료 밖 질문 답변이 어떻게 바뀌는지 확인합니다.
